# RGB-only Agentic Memory：从视频到推理与导航

本教程完整演示：

```text
RGB -> LingBot depth + c2w -> stable local submap -> spatial memory
RGB -> VLM semantics -> Pi -> Scene Graph -> Knowledge Memory
Knowledge Memory + Scene Graph -> Native Reasoner -> Navigation Action
```

运行时只依赖 RGB。GT、LiDAR 和 Isaac Sim 只用于离线评估。

## 1. 使用项目 Python 3.12 kernel

请在 VS Code notebook kernel 菜单中选择 **AgenticMemoryNav Python 3.12**。项目使用 `StrEnum`，Python 3.10 kernel 无法执行核心代码。

In [ ]:
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError('Select the AgenticMemoryNav Python 3.12 kernel, then rerun.')

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import numpy as np
print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)
print('Project root:', project_root)

## 2. LingBot depth+c2w 如何形成稳定 local submap

LingBot 的 depth head、camera head 和内参使每个 RGB frame 可以反投影为局部点云。机器人移动时，视野中心会变化，因此不能使用点云质心静止作为稳定条件。

当前 gate 使用相邻点云的对称最近邻残差：

$$r(P_t,P_{t+1})=\frac{1}{2}(d(P_t,P_{t+1})+d(P_{t+1},P_t))$$

窗口中的最大残差低于阈值时，stable RGB-only submap 会写入 `MemoryType.SPATIAL`，并保存 residual、confidence、frame IDs 和 provenance。

In [ ]:
from agentic_memory_nav.common.types import MappingUpdate, Pose3D
from agentic_memory_nav.mapping.local_submap import LocalSubmapBuilder

def mapping_update(index, offset_x):
    cloud = np.array([
        [offset_x, 0.0, 1.0],
        [offset_x + 0.02, 0.0, 1.0],
        [offset_x, 0.02, 1.0],
    ], dtype=np.float32)
    return MappingUpdate(
        frame_id=f'frame_{index:04d}', timestamp=float(index),
        camera_pose=Pose3D(position=(offset_x, 0.0, 0.0)),
        depth=np.ones((2, 2), dtype=np.float32),
        confidence=np.ones((2, 2), dtype=np.float32),
        local_pointcloud=cloud, global_pointcloud=cloud,
        is_keyframe=True, map_version=index + 1,
    )

builder = LocalSubmapBuilder(window_frames=3, frame_stride=1, stability_threshold_m=0.10)
submap = None
for index, offset_x in enumerate((0.00, 0.03, 0.06)):
    submap = builder.add(mapping_update(index, offset_x))

assert submap is not None
assert submap.stable
print('stable:', submap.stable)
print('frames:', submap.frame_ids)
print('overlap residual (m):', round(submap.geometric_residual_m, 4))
print('point count:', len(submap.points))

In [ ]:
unstable_builder = LocalSubmapBuilder(window_frames=3, frame_stride=1, stability_threshold_m=0.10)
unstable_builder.add(mapping_update(0, 0.00))
unstable_builder.add(mapping_update(1, 0.03))
unstable = unstable_builder.add(mapping_update(2, 2.00))

assert unstable is not None
assert not unstable.stable
print('stable:', unstable.stable)
print('overlap residual (m):', round(unstable.geometric_residual_m, 4))
print('decision: do not commit this geometry window')

## 3. VLM semantics + depth geometry 如何形成 $P_i$

VLM 给出类别、属性和 2D bbox。bbox fallback 或后续 SAM mask 从 depth 中选取像素，再通过 intrinsics 和 c2w 反投影为对象点云 $P_i$。

```text
VLM bbox -> instance mask -> depth pixels -> world points -> Pi NPZ artifact
```

未来可替换为开放式点云实例分割模型；$P_i$、graph 和 memory 的接口不需要重写。

In [ ]:
from agentic_memory_nav.common.types import CameraIntrinsics, FrameObservation, ObjectObservation
from agentic_memory_nav.geometry.pointcloud_store import PointCloudStore
from agentic_memory_nav.perception.instance_segmentation import BoundingBoxSegmenter, InstanceGeometryEnricher

rgb = np.zeros((48, 64, 3), dtype=np.uint8)
depth = np.full((48, 64), 2.0, dtype=np.float32)
frame = FrameObservation(
    frame_id='pi_frame', timestamp=0.0, rgb=rgb, depth=depth,
    camera_intrinsics=CameraIntrinsics(60.0, 60.0, 32.0, 24.0, 64, 48),
    camera_pose=Pose3D(),
)
mapping = mapping_update(0, 0.0)
mapping.depth = depth
mapping.confidence = np.ones_like(depth)
cube = ObjectObservation(
    observation_id='obs_red_cube', category='cube', attributes={'color': 'red'},
    bbox_2d=(20, 12, 44, 36), center_3d=(0.0, 0.0, 0.0),
    dimensions_3d=(0.0, 0.0, 0.0), confidence=0.9,
    timestamp=0.0, frame_id=frame.frame_id,
)

pi_store = PointCloudStore(Path('/tmp/agentic_memory_nav_workshop_v3_pi'))
cube = InstanceGeometryEnricher(BoundingBoxSegmenter(), pi_store).enrich(frame, mapping, [cube])[0]

assert cube.geometry is not None
print('Pi artifact:', cube.geometry.artifact_path)
print('Pi point count:', cube.geometry.point_count)
print('Pi centroid:', cube.geometry.centroid_3d)

## 4. Scene Graph -> Knowledge Memory

对象 $P_i$ 进入 Scene Graph 后成为 object node。room node 与 object node 通过 `inside` 等 relation edge 连接。`KnowledgeMemory` 再把这些节点和有向边 materialize 为 SQLite memory graph facts。

这样推理不是直接依赖一次 VLM 回答，而是依赖带 provenance 的历史 graph evidence。

In [ ]:
from agentic_memory_nav.memory.knowledge_memory import KnowledgeMemory
from agentic_memory_nav.memory.sqlite_store import SQLiteMemory
from agentic_memory_nav.scene_graph.graph import SceneGraph
from agentic_memory_nav.scene_graph.updater import SceneGraphUpdater

# room 与 cube 位置分开，association 不会把两个不同类别合并成同一个 node。
room = ObjectObservation(
    observation_id='obs_kitchen', category='kitchen', attributes={'kind': 'room'},
    bbox_2d=(0, 0, 64, 48), center_3d=(3.0, 0.0, 3.0),
    dimensions_3d=(6.0, 3.0, 6.0), confidence=0.99,
    timestamp=0.0, frame_id='pi_frame',
)
graph = SceneGraph()
SceneGraphUpdater(graph).update([room, cube])

memory_path = Path('/tmp/agentic_memory_nav_workshop_v3.sqlite3')
if memory_path.exists():
    memory_path.unlink()
memory = SQLiteMemory(memory_path)
knowledge = KnowledgeMemory(memory)
facts_created = knowledge.materialize(graph)

print('graph nodes:', [(node.label, node.node_type.value) for node in graph.nodes()])
print('graph edges:', [(edge.relation, round(edge.confidence, 2)) for edge in graph.edges()])
print('knowledge facts created:', facts_created)
print('memory graph query:', [item.content for item in knowledge.retrieve_subgraph('cube kitchen')])

## 5. Memory Graph -> Native Reasoning -> Navigation

这一步是 agent 的关键：

1. parser 从任务文本提取 object 和 room；
2. reasoner 在 graph/memory graph 中寻找 object node；
3. 它验证 `inside(object, room)` relation；
4. planner 根据 reasoning evidence 产生高层 `NAVIGATE` waypoint；
5. 若目标不存在，planner 产生 `EXPLORE`，而不是编造目标位置。

当前 MVP parser 稳定支持 `Find cube in the kitchen`。颜色等细粒度信息已保存在 graph node attributes，可由更强的结构化 parser 在未来利用。

In [ ]:
from agentic_memory_nav.planning.rule_based_fallback import RuleBasedPlanner
from agentic_memory_nav.planning.task_parser import RuleBasedTaskParser
from agentic_memory_nav.reasoning.native_reasoner import NativeReasoner

task = RuleBasedTaskParser().parse('Find cube in the kitchen')
reasoning = NativeReasoner(knowledge).resolve(graph, task.parsed_goal)
plan = RuleBasedPlanner(approach_distance=0.6).plan(
    task=task, robot_pose=Pose3D(), graph=graph, memory=memory,
    replan_reason='new stable RGB-only submap',
)

print('parsed goal:', task.parsed_goal)
print('reasoning target:', reasoning.target_id)
print('reasoning evidence:', reasoning.evidence_ids)
print('navigation action:', plan.action.action_type.value)
print('target node:', plan.action.target)
print('waypoint:', plan.action.waypoint)
print('confidence:', round(plan.confidence, 3))
print('information gaps:', plan.information_gaps)

assert plan.action.action_type.value == 'navigate'
assert plan.action.target == reasoning.target_id
assert plan.action.waypoint is not None

## 6. 实时推理：每个 RGB frame 都产生下一步 Navigation Action

前一节展示了已有 memory graph 上的一次 navigation plan。真实 agent 必须持续循环，而不是等任务结束才推理：

```text
new RGB frame
-> LingBot mapper 更新 depth + c2w + local submap
-> VLM / perception 输出当前 objects 与 triples
-> Pi、Scene Graph、Knowledge Memory 增量更新
-> NativeReasoner 查询最新 memory graph
-> Planner 立即输出 NAVIGATE / EXPLORE / VERIFY
-> 外部控制器执行该 action
```

`RealtimeAgent.ingest_frame(frame)` 就是这个单帧 runtime API。它不负责采集相机或底层电机控制；它负责把一帧观察转成最新 memory graph，并返回外部控制器下一步应执行的高层导航 action。

In [ ]:
from agentic_memory_nav.agent.realtime_agent import RealtimeAgent


def realtime_frame(index):
    return FrameObservation(
        frame_id=f"realtime_{index:04d}",
        timestamp=float(index),
        rgb=np.zeros((64, 96, 3), dtype=np.uint8),
        depth=np.full((64, 96), 2.0, dtype=np.float32),
        camera_intrinsics=CameraIntrinsics(80.0, 80.0, 48.0, 32.0, 96, 64),
        camera_pose=Pose3D(),
        robot_pose=Pose3D(),
    )


agent_root = Path("/tmp/agentic_memory_nav_realtime_workshop")
agent = RealtimeAgent(agent_root, "Find the red cup in the kitchen")
try:
    # Frame 0: mock perception observes only kitchen. The target is unknown, so explore.
    first = agent.ingest_frame(realtime_frame(0))
    print("frame 0 action:", first.plan.action.action_type.value)
    print("frame 0 graph nodes:", first.graph_nodes)
    print("frame 0 knowledge facts:", first.knowledge_facts_created)

    # Frame 1: the cup is now observed. Graph/memory update, then action changes immediately.
    second = agent.ingest_frame(realtime_frame(1))
    print("frame 1 action:", second.plan.action.action_type.value)
    print("frame 1 target:", second.plan.action.target)
    print("frame 1 waypoint:", second.plan.action.waypoint)
    print("frame 1 graph nodes:", second.graph_nodes)
    print("frame 1 knowledge facts:", second.knowledge_facts_created)

    assert first.plan.action.action_type.value == "explore"
    assert second.plan.action.action_type.value == "navigate"
    assert second.plan.action.waypoint is not None
finally:
    agent.close()

### 没有 memory graph 证据：Explore

这是 agent 的认识论约束：目标未被观察到时，先安全探索并收集新的 RGB、LingBot local submap 和 VLM 语义证据，而不是编造导航位置。

In [ ]:
empty_graph = SceneGraph()
empty_path = Path('/tmp/agentic_memory_nav_workshop_v3_empty.sqlite3')
if empty_path.exists():
    empty_path.unlink()
empty_memory = SQLiteMemory(empty_path)
explore_plan = RuleBasedPlanner().plan(
    task=task, robot_pose=Pose3D(), graph=empty_graph, memory=empty_memory
)

print('navigation action:', explore_plan.action.action_type.value)
print('exploration waypoint:', explore_plan.action.waypoint)
print('information gaps:', explore_plan.information_gaps)
assert explore_plan.action.action_type.value == 'explore'
assert explore_plan.replan_required

empty_memory.close()
memory.close()

## 7. Runtime 与离线评测的边界

运行时：

```text
new RGB frame -> LingBot depth/c2w -> stable local submap -> spatial memory
new RGB frame -> VLM objects/triples -> scene graph + knowledge memory
updated memory graph -> native reasoning -> NAVIGATE / EXPLORE / VERIFY
```

每一帧的 action 都来自最新 graph/memory state，而不是固定脚本。外部控制器执行当前 action 后，下一帧又会触发一次新的推理与重规划。

GT depth、GT c2w 与 LiDAR 仅用于离线评测。stable RGB-only submap 会携带 overlap residual、confidence 与 provenance 写入 memory；低置信度、冲突关系或 room evidence 缺失时，agent 应重新观察或执行 `VERIFY`。